# 🔥 CodeBlaze — AI Resume Screening System
## Exploratory Data Analysis & Model Evaluation Notebook

**PRD**: AI-Powered Resume Screening System — ATS Score Prediction & Resume Enhancement

**Dataset**: Mirrors Kaggle `UpdatedResumeDataSet` (8 categories, 500+ resumes)

### Notebook Structure
1. Data Loading & Overview
2. EDA — Resume Category Distribution
3. EDA — ATS Score Analysis
4. Text Analysis — TF-IDF Keyword Exploration
5. ATS Scorer Training & Evaluation
6. Resume Classifier Training & Evaluation
7. End-to-End Demo

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print('Libraries loaded ✓')

## 1. Data Loading & Overview

In [ ]:
from data.generate_dataset import generate_resumes, generate_job_descriptions

# Generate dataset (or load from CSV)
df = generate_resumes(500)
df_jd = generate_job_descriptions()

print(f'Resume dataset shape: {df.shape}')
print(f'JD dataset shape: {df_jd.shape}')
display(df.head(3))

In [ ]:
# Basic stats
print('\nDataset Info:')
print(df.dtypes)
print('\nATS Score Summary:')
print(df['ats_score'].describe().round(2))

## 2. EDA — Category Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Category counts
cat_counts = df['category'].value_counts()
axes[0].barh(cat_counts.index, cat_counts.values, color='steelblue', edgecolor='white')
axes[0].set_title('Resume Count by Category', fontweight='bold')
axes[0].set_xlabel('Count')
for i, v in enumerate(cat_counts.values):
    axes[0].text(v + 0.5, i, str(v), va='center')

# Quality distribution per category
quality_data = df.groupby(['category', 'quality']).size().unstack(fill_value=0)
quality_data.plot(kind='barh', stacked=True, ax=axes[1],
                  color=['#E74C3C', '#F39C12', '#2ECC71'])
axes[1].set_title('Resume Quality Distribution by Category', fontweight='bold')
axes[1].legend(title='Quality')

plt.tight_layout()
plt.show()

## 3. EDA — ATS Score Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# ATS score distribution
axes[0,0].hist(df['ats_score'], bins=30, color='#4F8EF7', edgecolor='white', alpha=0.85)
axes[0,0].set_title('Overall ATS Score Distribution', fontweight='bold')
axes[0,0].set_xlabel('ATS Score')
axes[0,0].axvline(df['ats_score'].mean(), color='red', linestyle='--',
                  label=f"Mean: {df['ats_score'].mean():.1f}")
axes[0,0].legend()

# Score by category boxplot
df.boxplot(column='ats_score', by='category', ax=axes[0,1], rot=45)
axes[0,1].set_title('ATS Score by Category', fontweight='bold')
axes[0,1].set_xlabel('')

# Score by quality
df.groupby('quality')['ats_score'].hist(bins=20, ax=axes[1,0], alpha=0.7,
                                         label=['high','low','medium'])
axes[1,0].set_title('ATS Score Distribution by Resume Quality', fontweight='bold')
axes[1,0].legend(['high', 'low', 'medium'])

# Experience vs ATS Score
axes[1,1].scatter(df['experience_years'], df['ats_score'],
                  c=df['ats_score'], cmap='RdYlGn', alpha=0.6, edgecolor='none')
axes[1,1].set_title('Experience vs ATS Score', fontweight='bold')
axes[1,1].set_xlabel('Years of Experience')
axes[1,1].set_ylabel('ATS Score')

plt.tight_layout()
plt.show()

## 4. Text Analysis — TF-IDF Keyword Exploration

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# Top keywords per category
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Top TF-IDF Keywords per Category', fontsize=14, fontweight='bold')

colors = ['#4F8EF7','#34C785','#F7954F','#E74C3C','#9B59B6','#1ABC9C','#F39C12','#2ECC71']

for idx, (category, group) in enumerate(df.groupby('category')):
    if idx >= 8:
        break
    vec = TfidfVectorizer(ngram_range=(1,2), stop_words='english', max_features=500)
    tfidf = vec.fit_transform(group['resume_text'])
    mean_scores = tfidf.mean(axis=0).A1
    top_idx = mean_scores.argsort()[-10:][::-1]
    top_words = [vec.get_feature_names_out()[i] for i in top_idx]
    top_scores = [mean_scores[i] for i in top_idx]
    
    ax = axes[idx // 4][idx % 4]
    ax.barh(top_words[::-1], top_scores[::-1], color=colors[idx], alpha=0.85)
    ax.set_title(category, fontweight='bold', fontsize=10)
    ax.set_xlabel('TF-IDF Score', fontsize=8)
    ax.tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.show()

## 5. ATS Scorer Training & Evaluation

In [ ]:
from models.ats_scorer import ATSScorer, HeuristicScorer
from tqdm.notebook import tqdm

# Extract heuristic features
heuristic = HeuristicScorer()
df['formatting_score'] = df['resume_text'].apply(heuristic.score_formatting)
df['completeness_score'] = df['resume_text'].apply(heuristic.score_completeness)
df['readability_score'] = df['resume_text'].apply(heuristic.score_readability)
df['action_verb_score'] = df['resume_text'].apply(heuristic.score_action_verbs)

scorer = ATSScorer()
scorer.fit_tfidf(df['resume_text'].tolist())
df['keyword_score'] = df['resume_text'].apply(scorer._general_keyword_density)

print('Features extracted!')
display(df[['formatting_score','completeness_score','readability_score',
            'action_verb_score','keyword_score','ats_score']].describe().round(2))

In [ ]:
# Train meta-learner
ats_metrics = scorer.train_meta_model(df)
print(f"\nBest Model: {ats_metrics['best_model']}")
print(f"Train R²: {ats_metrics['train_r2']:.4f}")
print(f"Test  R²: {ats_metrics['test_r2']:.4f}")

In [ ]:
# Correlation heatmap
score_cols = ['keyword_score','formatting_score','completeness_score',
              'readability_score','action_verb_score','ats_score']
corr = df[score_cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', square=True,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('ATS Sub-Score Correlation Matrix', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Resume Classifier Training & Evaluation

In [ ]:
from models.resume_classifier import ResumeClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

clf = ResumeClassifier()
clf_metrics = clf.train(df)

print(f"\nCV Accuracy:   {clf_metrics['cv_accuracy']:.4f}")
print(f"Test Accuracy: {clf_metrics['test_accuracy']:.4f}")

In [ ]:
# Per-class metrics
report = clf_metrics['classification_report']
classes = clf_metrics['classes']

f1_scores = [report[c]['f1-score'] for c in classes if c in report]
precision = [report[c]['precision'] for c in classes if c in report]
recall = [report[c]['recall'] for c in classes if c in report]

x = np.arange(len(classes))
width = 0.25

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(x - width, precision, width, label='Precision', color='#4F8EF7', alpha=0.85)
ax.bar(x, recall, width, label='Recall', color='#34C785', alpha=0.85)
ax.bar(x + width, f1_scores, width, label='F1-Score', color='#F7954F', alpha=0.85)

ax.set_xlabel('Category')
ax.set_ylabel('Score')
ax.set_title('Classifier Metrics per Category', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(classes, rotation=30, ha='right')
ax.legend()
ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.show()

## 7. End-to-End Demo

In [ ]:
from models.keyword_gap_analyzer import KeywordGapAnalyzer
from models.company_recommender import CompanyRecommender

sample_resume = """
Priya Sharma | priya@email.com | linkedin.com/in/priyasharma

SUMMARY
B.Tech CSE final year student seeking placement at top IT companies.

SKILLS
Python, SQL, Java, HTML, CSS, Git

EXPERIENCE
Worked on web development project using HTML and CSS.
Helped team with data entry and documentation tasks.

EDUCATION
B.Tech Computer Science, VTU, 2024 | CGPA: 8.5
"""

sample_jd = """
Junior Software Engineer — Required: Python, Java, SQL, REST API, Git, Agile.
Preferred: Docker, AWS, Spring Boot, Microservices.
Experience: 0-2 years or fresh graduates welcome.
"""

# Score
result = scorer.score(sample_resume, sample_jd)
print(f'Overall ATS Score: {result.overall_score}/100 [{result.grade()}]')
print(f'  Keyword:      {result.keyword_score:.1f}')
print(f'  Formatting:   {result.formatting_score:.1f}')
print(f'  Completeness: {result.completeness_score:.1f}')
print(f'  Readability:  {result.readability_score:.1f}')
print(f'  Action Verbs: {result.action_verb_score:.1f}')

In [ ]:
# Radar chart of sub-scores
import numpy as np
import matplotlib.pyplot as plt

categories = ['Keyword', 'Formatting', 'Completeness', 'Readability', 'Action Verbs']
values = [
    result.keyword_score, result.formatting_score, result.completeness_score,
    result.readability_score, result.action_verb_score
]

angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
values_plot = values + [values[0]]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.plot(angles, values_plot, 'o-', linewidth=2, color='#4F8EF7')
ax.fill(angles, values_plot, alpha=0.25, color='#4F8EF7')
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, size=12)
ax.set_ylim(0, 100)
ax.set_yticks([20, 40, 60, 80, 100])
ax.set_title(f'ATS Sub-Score Radar\nOverall: {result.overall_score}/100',
             size=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Improvement tips
print('\nTop Improvement Tips:')
for i, tip in enumerate(result.improvement_tips, 1):
    print(f'{i}. [{tip["priority"]}] {tip["category"]}')
    print(f'   Issue: {tip["issue"]}')
    print(f'   Fix:   {tip["recommendation"]}')
    print(f'   Gain:  {tip["expected_improvement"]}\n')

In [ ]:
# Company fit
recommender = CompanyRecommender()
fit = recommender.compare_companies(sample_resume, recommender.list_companies())

companies = list(fit.keys())
scores = [fit[c]['fit_score'] for c in companies]

colors = ['#2ECC71' if s >= 50 else '#F39C12' if s >= 30 else '#E74C3C' for s in scores]

plt.figure(figsize=(12, 5))
bars = plt.barh(companies, scores, color=colors, edgecolor='white')
plt.xlabel('Company Fit Score (%)')
plt.title('Resume Fit Score by Target Company', fontweight='bold')
plt.xlim(0, 110)
for bar, score in zip(bars, scores):
    plt.text(score + 1, bar.get_y() + bar.get_height()/2,
             f'{score:.0f}%', va='center', fontweight='bold')
plt.tight_layout()
plt.show()